### RNN Model 

In [17]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense,Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### load the datasets

In [2]:
X_train_padded = np.load("../models/X_train_padded.npy")
X_val_padded = np.load("../models/X_val_padded.npy")
X_test_padded = np.load("../models/X_test_padded.npy")
y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

sequence_length = X_train_padded.shape[1]
vocab_size = 20000

print(X_train_padded.shape)
print(X_val_padded.shape)
print(X_test_padded.shape)

(34705, 200)
(7439, 200)
(7438, 200)


In [19]:
rnn_results = []

def evaluate_rnn(model, experiment_name, X_test_data=None):

    # Use 200-token test data by default
    if X_test_data is None:
        X_test_data = X_test_padded

    # Prediction probabilities
    y_prob = model.predict(
        X_test_data,
        verbose=0
    ).ravel()

    # Convert probabilities to classes
    y_pred = (y_prob >= 0.5).astype(int)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    rnn_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    # Display results
    print(f"\n{experiment_name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

### Baseline RNN

In [4]:
rnn_model = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

E0000 00:00:1787566786.326366   61188 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787566786.326854   61773 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787566786.342234   61188 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,576,577 (9.83 MB)

 Trainable params: 2,576,577 (9.83 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history_rnn = rnn_model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 103s 93ms/step - accuracy: 0.4963 - loss: 0.6988 - val_accuracy: 0.5069 - val_loss: 0.6953
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 105s 97ms/step - accuracy: 0.4998 - loss: 0.6958 - val_accuracy: 0.5040 - val_loss: 0.6951
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.5154 - loss: 0.6889 - val_accuracy: 0.5092 - val_loss: 0.6942
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 104s 96ms/step - accuracy: 0.5418 - loss: 0.6626 - val_accuracy: 0.5019 - val_loss: 0.7063
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 106s 98ms/step - accuracy: 0.5539 - loss: 0.6369 - val_accuracy: 0.5114 - val_loss: 0.7175


In [20]:
evaluate_rnn(
    rnn_model,
    "RNN - Baseline"
)
rnn_model.save("../models/rnn_baseline.keras")


RNN - Baseline
----------------------------------------
Accuracy : 0.5136
Precision: 0.5917
Recall   : 0.0994
F1 Score : 0.1702
ROC-AUC  : 0.5198


### Embedding Dimension

In [7]:
rnn_embedding = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=256
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_embedding.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [8]:
history_embedding = rnn_embedding.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 151s 137ms/step - accuracy: 0.5052 - loss: 0.6975 - val_accuracy: 0.4968 - val_loss: 0.6957
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 147s 135ms/step - accuracy: 0.5076 - loss: 0.6945 - val_accuracy: 0.5049 - val_loss: 0.6972
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 147s 136ms/step - accuracy: 0.5232 - loss: 0.6855 - val_accuracy: 0.5030 - val_loss: 0.6969
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 164s 151ms/step - accuracy: 0.5415 - loss: 0.6678 - val_accuracy: 0.5161 - val_loss: 0.7085
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 153s 141ms/step - accuracy: 0.5429 - loss: 0.6528 - val_accuracy: 0.5032 - val_loss: 0.7217


In [21]:
evaluate_rnn(
    rnn_embedding,
    "RNN - Embedding 256"
)

rnn_embedding.save("../models/rnn_embedding_256.keras")


RNN - Embedding 256
----------------------------------------
Accuracy : 0.5009
Precision: 0.5024
Recall   : 0.5773
F1 Score : 0.5373
ROC-AUC  : 0.5098


### Sequential Length

In [25]:
with open("../models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

vocab_size = len(tokenizer.word_index) + 1

In [26]:
X_train_text = pd.read_pickle("../models/X_train_text.pkl")
X_val_text = pd.read_pickle("../models/X_val_text.pkl")
X_test_text = pd.read_pickle("../models/X_test_text.pkl")

print(X_train_text.shape)
print(X_val_text.shape)
print(X_test_text.shape)

(34705,)
(7439,)
(7438,)


In [27]:
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_val_sequences = tokenizer.texts_to_sequences(X_val_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_padded_300 = pad_sequences(
    X_train_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_val_padded_300 = pad_sequences(
    X_val_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_test_padded_300 = pad_sequences(
    X_test_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

In [24]:
rnn_sequence = Sequential([
    Input(shape=(300,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_sequence.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [25]:
history_sequence = rnn_sequence.fit(
    X_train_padded_300,
    y_train,
    validation_data=(X_val_padded_300, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 291s 267ms/step - accuracy: 0.5001 - loss: 0.7000 - val_accuracy: 0.4982 - val_loss: 0.6966
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 280s 258ms/step - accuracy: 0.5039 - loss: 0.6964 - val_accuracy: 0.5026 - val_loss: 0.6979
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 236s 218ms/step - accuracy: 0.5095 - loss: 0.6950 - val_accuracy: 0.5122 - val_loss: 0.6932
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 161s 148ms/step - accuracy: 0.5155 - loss: 0.6935 - val_accuracy: 0.5057 - val_loss: 0.6955
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 162s 149ms/step - accuracy: 0.5127 - loss: 0.6883 - val_accuracy: 0.5010 - val_loss: 0.6980


In [29]:
evaluate_rnn(
    rnn_sequence,
    "RNN - Sequence Length 300",
    X_test_padded_300
)
rnn_sequence.save("../models/rnn_sequence_300.keras")


RNN - Sequence Length 300
----------------------------------------
Accuracy : 0.5028
Precision: 0.5829
Recall   : 0.0329
F1 Score : 0.0624
ROC-AUC  : 0.5033


### Hidden Units

In [37]:
rnn_hidden = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        128,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_hidden.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [39]:
history_hidden = rnn_hidden.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 139ms/step - accuracy: 0.5547 - loss: 0.6380 - val_accuracy: 0.5251 - val_loss: 0.7306
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 145s 134ms/step - accuracy: 0.5686 - loss: 0.6179 - val_accuracy: 0.5175 - val_loss: 0.7543
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 142s 131ms/step - accuracy: 0.5725 - loss: 0.6049 - val_accuracy: 0.5169 - val_loss: 0.8049
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 147s 135ms/step - accuracy: 0.5726 - loss: 0.5999 - val_accuracy: 0.5229 - val_loss: 0.8540
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 134ms/step - accuracy: 0.5728 - loss: 0.5990 - val_accuracy: 0.5192 - val_loss: 0.8565


In [30]:
evaluate_rnn(
    rnn_hidden,
    "RNN - Hidden Units 128"
)

rnn_hidden.save("../models/rnn_hidden_128.keras")


RNN - Hidden Units 128
----------------------------------------
Accuracy : 0.5171
Precision: 0.6189
Recall   : 0.0983
F1 Score : 0.1697
ROC-AUC  : 0.5421


### Dropout

In [6]:
rnn_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [7]:
history_dropout = rnn_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 87s 58ms/step - accuracy: 0.5002 - loss: 0.7030 - val_accuracy: 0.5005 - val_loss: 0.6962
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 62s 58ms/step - accuracy: 0.4970 - loss: 0.6968 - val_accuracy: 0.5005 - val_loss: 0.6932
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 67s 62ms/step - accuracy: 0.5042 - loss: 0.6940 - val_accuracy: 0.5122 - val_loss: 0.6927
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 90s 83ms/step - accuracy: 0.5082 - loss: 0.6934 - val_accuracy: 0.5091 - val_loss: 0.6931
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 103s 95ms/step - accuracy: 0.5112 - loss: 0.6927 - val_accuracy: 0.5072 - val_loss: 0.6931


In [31]:
evaluate_rnn(
    rnn_dropout,
    "RNN - Dropout"
)

rnn_dropout.save("../models/rnn_dropout.keras")


RNN - Dropout
----------------------------------------
Accuracy : 0.5051
Precision: 0.5312
Recall   : 0.1187
F1 Score : 0.1940
ROC-AUC  : 0.5050


### Different Optimizer

In [13]:
rnn_rmsprop = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [14]:

history_rmsprop = rnn_rmsprop.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 96s 87ms/step - accuracy: 0.5006 - loss: 0.6985 - val_accuracy: 0.5064 - val_loss: 0.6959
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 91s 84ms/step - accuracy: 0.5017 - loss: 0.6983 - val_accuracy: 0.5010 - val_loss: 0.6953
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 100s 92ms/step - accuracy: 0.5060 - loss: 0.6962 - val_accuracy: 0.5193 - val_loss: 0.6927
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 94s 87ms/step - accuracy: 0.5064 - loss: 0.6936 - val_accuracy: 0.5103 - val_loss: 0.6938
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 94s 87ms/step - accuracy: 0.5315 - loss: 0.6774 - val_accuracy: 0.5025 - val_loss: 0.7037


In [32]:

evaluate_rnn(
    rnn_rmsprop,
    "RNN - RMSprop"
)

rnn_rmsprop.save("../models/rnn_rmsprop.keras")


RNN - RMSprop
----------------------------------------
Accuracy : 0.5087
Precision: 0.5057
Recall   : 0.9432
F1 Score : 0.6584
ROC-AUC  : 0.5236


### Batch Normalization

In [34]:
rnn_batchnorm = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    BatchNormalization(),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_batchnorm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [36]:

history_batchnorm = rnn_batchnorm.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 254s 234ms/step - accuracy: 0.5065 - loss: 0.6947 - val_accuracy: 0.5025 - val_loss: 0.7575
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 257s 236ms/step - accuracy: 0.5334 - loss: 0.6685 - val_accuracy: 0.4931 - val_loss: 0.7062
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 252s 232ms/step - accuracy: 0.5152 - loss: 0.6944 - val_accuracy: 0.5032 - val_loss: 0.6942
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 262s 241ms/step - accuracy: 0.5397 - loss: 0.6829 - val_accuracy: 0.4981 - val_loss: 0.7095
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 263s 242ms/step - accuracy: 0.5582 - loss: 0.6595 - val_accuracy: 0.5313 - val_loss: 0.7205


In [37]:

evaluate_rnn(
    rnn_batchnorm,
    "RNN - Batch Normalization"
)

rnn_batchnorm.save("../models/rnn_batchnorm.keras")


RNN - Batch Normalization
----------------------------------------
Accuracy : 0.5379
Precision: 0.5671
Recall   : 0.3351
F1 Score : 0.4213
ROC-AUC  : 0.5465


### Learning Rate

In [38]:
rnn_learning_rate = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_learning_rate.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [39]:

history_learning_rate = rnn_learning_rate.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)



Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 255s 233ms/step - accuracy: 0.5010 - loss: 0.6937 - val_accuracy: 0.5107 - val_loss: 0.6929
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 251s 232ms/step - accuracy: 0.5506 - loss: 0.6801 - val_accuracy: 0.5075 - val_loss: 0.6962
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 256s 236ms/step - accuracy: 0.6367 - loss: 0.6243 - val_accuracy: 0.5337 - val_loss: 0.7204
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 260s 240ms/step - accuracy: 0.7841 - loss: 0.4539 - val_accuracy: 0.5325 - val_loss: 0.8238
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 256s 236ms/step - accuracy: 0.9207 - loss: 0.2127 - val_accuracy: 0.5366 - val_loss: 1.1265


In [40]:
evaluate_rnn(
    rnn_learning_rate,
    "RNN - Learning Rate 0.0001"
)

rnn_learning_rate.save("../models/rnn_learning_rate.keras")


RNN - Learning Rate 0.0001
----------------------------------------
Accuracy : 0.5242
Precision: 0.5267
Recall   : 0.5122
F1 Score : 0.5194
ROC-AUC  : 0.5415


### Batch Size

In [41]:
rnn_batch_size = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_batch_size.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [42]:

history_batch_size = rnn_batch_size.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)


Epoch 1/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 136s 247ms/step - accuracy: 0.5026 - loss: 0.6941 - val_accuracy: 0.5025 - val_loss: 0.6930
Epoch 2/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 134s 247ms/step - accuracy: 0.5484 - loss: 0.6620 - val_accuracy: 0.4948 - val_loss: 0.7232
Epoch 3/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 135s 248ms/step - accuracy: 0.5788 - loss: 0.5943 - val_accuracy: 0.5073 - val_loss: 0.7868
Epoch 4/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 112s 206ms/step - accuracy: 0.5848 - loss: 0.5842 - val_accuracy: 0.5044 - val_loss: 0.8096
Epoch 5/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 80s 147ms/step - accuracy: 0.5863 - loss: 0.5928 - val_accuracy: 0.5006 - val_loss: 0.7845


In [43]:

evaluate_rnn(
    rnn_batch_size,
    "RNN - Batch Size 64"
)

rnn_batch_size.save("../models/rnn_batch_size_64.keras")


RNN - Batch Size 64
----------------------------------------
Accuracy : 0.5013
Precision: 0.5018
Recall   : 0.9041
F1 Score : 0.6454
ROC-AUC  : 0.5072


### Early Stopping

In [44]:
rnn_early_stopping = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_early_stopping.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


In [45]:

history_early_stopping = rnn_early_stopping.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 154s 141ms/step - accuracy: 0.4981 - loss: 0.7001 - val_accuracy: 0.5057 - val_loss: 0.6967
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 152s 140ms/step - accuracy: 0.5091 - loss: 0.6966 - val_accuracy: 0.5089 - val_loss: 0.6942
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 151s 139ms/step - accuracy: 0.5168 - loss: 0.6911 - val_accuracy: 0.5122 - val_loss: 0.6945
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 154s 142ms/step - accuracy: 0.5381 - loss: 0.6741 - val_accuracy: 0.5147 - val_loss: 0.7016


In [46]:
evaluate_rnn(
    rnn_early_stopping,
    "RNN - Early Stopping"
)

rnn_early_stopping.save(
    "../models/rnn_early_stopping.keras"
)


RNN - Early Stopping
----------------------------------------
Accuracy : 0.5090
Precision: 0.5116
Recall   : 0.4766
F1 Score : 0.4935
ROC-AUC  : 0.5104


### Learning Rate Scheduling

In [47]:
rnn_scheduler = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_scheduler.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)


In [48]:

history_scheduler = rnn_scheduler.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 154s 141ms/step - accuracy: 0.5013 - loss: 0.6958 - val_accuracy: 0.5010 - val_loss: 0.6956 - learning_rate: 0.0010
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 153s 141ms/step - accuracy: 0.5274 - loss: 0.6841 - val_accuracy: 0.4976 - val_loss: 0.7177 - learning_rate: 0.0010
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 153s 141ms/step - accuracy: 0.5657 - loss: 0.6184 - val_accuracy: 0.4981 - val_loss: 0.7872 - learning_rate: 5.0000e-04
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 161s 149ms/step - accuracy: 0.5795 - loss: 0.5889 - val_accuracy: 0.5091 - val_loss: 0.8072 - learning_rate: 2.5000e-04
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 157s 145ms/step - accuracy: 0.5904 - loss: 0.5800 - val_accuracy: 0.5083 - val_loss: 0.8421 - learning_rate: 1.2500e-04


In [49]:

evaluate_rnn(
    rnn_scheduler,
    "RNN - Learning Rate Scheduling"
)

rnn_scheduler.save(
    "../models/rnn_learning_rate_scheduler.keras"
)


RNN - Learning Rate Scheduling
----------------------------------------
Accuracy : 0.5063
Precision: 0.5413
Recall   : 0.1072
F1 Score : 0.1789
ROC-AUC  : 0.5001


In [51]:
rnn_recurrent_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        64,
        activation="tanh",
        recurrent_dropout=0.3
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_recurrent_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [53]:
history_recurrent_dropout = rnn_recurrent_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 164s 151ms/step - accuracy: 0.4976 - loss: 0.6994 - val_accuracy: 0.5005 - val_loss: 0.6944
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 173s 159ms/step - accuracy: 0.5006 - loss: 0.6971 - val_accuracy: 0.4968 - val_loss: 0.6943
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 181s 167ms/step - accuracy: 0.5016 - loss: 0.6953 - val_accuracy: 0.5068 - val_loss: 0.6941
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 154s 142ms/step - accuracy: 0.5044 - loss: 0.6943 - val_accuracy: 0.5142 - val_loss: 0.6929
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 152s 140ms/step - accuracy: 0.5054 - loss: 0.6938 - val_accuracy: 0.5015 - val_loss: 0.6929


In [54]:

evaluate_rnn(
    rnn_recurrent_dropout,
    "RNN - Recurrent Dropout"
)

rnn_recurrent_dropout.save(
    "../models/rnn_recurrent_dropout.keras"
)


RNN - Recurrent Dropout
----------------------------------------
Accuracy : 0.5012
Precision: 0.5016
Recall   : 0.9917
F1 Score : 0.6662
ROC-AUC  : 0.4994


In [55]:
rnn_results_df = pd.DataFrame(rnn_results)

rnn_results_df

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,RNN - Baseline,0.513579,0.591707,0.099384,0.170183,0.519775
1,RNN - Embedding 256,0.500941,0.502448,0.577284,0.537273,0.509831
2,RNN - Sequence Length 300,0.502823,0.582938,0.032949,0.062373,0.503281
3,RNN - Hidden Units 128,0.517074,0.618887,0.098312,0.169672,0.542050
4,RNN - Dropout,0.505109,0.531175,0.118671,0.194000,0.504956
5,RNN - RMSprop,0.508739,0.505673,0.943209,0.658377,0.523570
6,RNN - Batch Normalization,0.537913,0.567090,0.335119,0.421283,0.546459
7,RNN - Learning Rate 0.0001,0.524200,0.526722,0.512189,0.519354,0.541466
8,RNN - Batch Size 64,0.501344,0.501784,0.904099,0.645377,0.507164
9,RNN - Early Stopping,0.509008,0.511648,0.476560,0.493481,0.510435


In [56]:
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
Pandas: 3.0.5
NumPy: 2.5.2


In [57]:
X_train_text.to_csv("../models/X_train_text.csv", index=False)
X_val_text.to_csv("../models/X_val_text.csv", index=False)
X_test_text.to_csv("../models/X_test_text.csv", index=False)